<a href="https://colab.research.google.com/github/Yassman04/bert-ner-pipeline/blob/main/ner_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition with BERT

**Author:** Yaseen Mneimneih

An end-to-end pipeline that finds people, organisations, locations and other named entities in text. It compares a simple rule-based baseline with a fine-tuned `bert-base-cased` model on the CoNLL-2003 dataset, then tests the model on real sentences and analyses its mistakes.

Run this notebook in Google Colab (GPU runtime recommended) or any Jupyter environment with a GPU.

In [1]:
# seqeval needs these build tools to install on newer Python versions (3.13+)
!pip install -q "setuptools<81" wheel setuptools_scm
!pip install -q --no-build-isolation seqeval
!pip install -q transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### 1. Data Ingestion
The primary dataset for this pipeline is the CoNLL 2003 shared task corpus. We will load this directly using the Hugging Face datasets library. This provides the data in a convenient format with predefined train, validation, and test splits.

In [4]:
from datasets import load_dataset

# Load the modern, secure version of CoNLL-2003 to bypass script blocks
dataset = load_dataset("lhoestq/conll2003")

# Print the dataset dictionary to verify the splits
print(dataset)

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


### 2. Initial Data Inspection
Before building the tokenization logic, it is essential to understand the raw data structure. We will extract the first sentence from the training set to observe how the individual tokens and their corresponding Named Entity Recognition integer tags are currently formatted.

In [5]:
# Extract the first instance in the training set
example = dataset["train"][0]

# Display the raw tokens and their corresponding integer tags
print("Tokens:", example["tokens"])
print("NER Tags:", example["ner_tags"])

Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
NER Tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]


### 3. Tokenization and Label Alignment
Transformer models like BERT utilize sub-word tokenization. This creates a misalignment between our original word-level IOB2 tags and the newly generated tokens. To resolve this, we will write a function that expands our label arrays to match the new token lengths, assigning a label of `-100` to special tokens and sub-word fragments so they are ignored by the PyTorch loss function during training.

In [6]:
from transformers import AutoTokenizer

# Initialize the fast tokenizer for the BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [7]:
def tokenize_and_align_labels(examples):
    # Tokenize the input, ensuring it knows the text is already split into words
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    # Iterate through each sentence and its corresponding NER tags
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens like [CLS] and [SEP] get a word_id of None, we assign them -100
            if word_idx is None:
                label_ids.append(-100)
            # Assign the true label to the very first sub-word of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Assign -100 to all subsequent sub-word fragments of that same word
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the alignment function to the entire train, validation, and test dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


print("Tokenization and alignment complete!")
print("New Token IDs:", tokenized_dataset["train"][0]["input_ids"])
print("Aligned Labels:", tokenized_dataset["train"][0]["labels"])

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenization and alignment complete!
New Token IDs: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Aligned Labels: [-100, 3, 0, 7, 0, 0, 0, 7, 0, -100, 0, -100]


### 4. Heuristic Baseline Model
To establish a benchmark for our neural network, we first implement a heuristic-based baseline. This rule-based approach categorizes entities by flagging any capitalized word not found at the start of a sentence as a potential entity (categorized temporarily as `B-MISC`). We will evaluate this baseline using the `seqeval` library to quantify the contextual gain our deep learning model provides later.

In [8]:
from seqeval.metrics import classification_report, f1_score

label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

def heuristic_baseline(dataset_split):
    true_labels = []
    predictions = []

    for example in dataset_split:
        tokens = example["tokens"]
        labels = [label_list[tag] for tag in example["ner_tags"]]
        preds = []

        for i, token in enumerate(tokens):
            if i > 0 and token[0].isupper():
                preds.append("B-MISC")
            else:
                preds.append("O")

        true_labels.append(labels)
        predictions.append(preds)

    print("--- Heuristic Baseline Results ---")
    print(classification_report(true_labels, predictions))
    return f1_score(true_labels, predictions)

baseline_f1 = heuristic_baseline(dataset["test"])
print(f"Baseline Overall F1 Score: {baseline_f1:.4f}")

--- Heuristic Baseline Results ---


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00      1668
        MISC       0.05      0.61      0.10       702
         ORG       0.00      0.00      0.00      1661
         PER       0.00      0.00      0.00      1617

   micro avg       0.05      0.08      0.06      5648
   macro avg       0.01      0.15      0.02      5648
weighted avg       0.01      0.08      0.01      5648

Baseline Overall F1 Score: 0.0618


### 5. Data Formatting and Padding
Deep learning models require inputs of uniform length for batch processing. We utilize a Hugging Face `DataCollatorForTokenClassification` to dynamically pad our sequences during training. We also strip out unnecessary string columns and convert our dataset format directly to PyTorch tensors.

In [9]:
from transformers import DataCollatorForTokenClassification

# The data collator handles dynamic padding for PyTorch batches
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Remove columns that the BERT model does not need for training
columns_to_remove = ["id", "tokens", "pos_tags", "chunk_tags", "ner_tags"]
formatted_dataset = tokenized_dataset.remove_columns(columns_to_remove)

# Convert the datasets to PyTorch tensors
formatted_dataset.set_format("torch")

print("Data formatted for PyTorch successfully!")

Data formatted for PyTorch successfully!


### 6. Model Initialization
With our data prepared, we initialize the deep learning architecture. We use the `bert-base-cased` model from Hugging Face, specifically configured for token classification. We pass the exact number of labels and our ID to label mappings so the final classification layer is sized correctly for our 9 CoNLL 2003 entity tags.

In [10]:
from transformers import AutoModelForTokenClassification

# Standard CoNLL 2003 labels
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

# Create dictionaries mapping IDs to Labels and Labels to IDs
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Initialize the BERT model for token classification
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"BERT model loaded and configured for {len(label_list)} classes!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

BERT model loaded and configured for 9 classes!


### 7. Evaluation Metrics
During training, the model needs to evaluate its performance on the validation set to ensure it is actually learning and not just memorizing (overfitting). We define a `compute_metrics` function that takes the model's raw predictions, ignores the `-100` padding/special tokens we set up earlier, and uses `seqeval` to calculate the Precision, Recall, F1, and Accuracy scores.

In [11]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, accuracy_score

def compute_metrics(p):
    predictions, labels = p
    # Get the highest probability prediction for each token
    predictions = np.argmax(predictions, axis=2)

    # Filter out the -100 labels we used for special tokens/sub-words
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

print("Metrics function defined!")

Metrics function defined!


### 8. Model Training (TRAINS FROM SCRATCH)
We now configure the `Trainer` object. We will train the model for 3 epochs with a learning rate of 2e-5. At the end of each epoch, the model will evaluate itself against the validation set using our custom metrics function. This is the computationally heavy part of the pipeline.

In [ ]:
from transformers import TrainingArguments, Trainer

# Define the training hyperparameters
training_args = TrainingArguments(
    output_dir="./bert-ner-results",
    eval_strategy="epoch",
    learning_rate=2e-5,           # Standard learning rate for fine-tuning BERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,           # 3 passes through the data
    weight_decay=0.01,
    logging_steps=100
)

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting the training loop...")
# Kick off the training!
trainer.train()

### 8.1 Load the Fine-Tuned Model (skip training)
Training takes around 15 minutes on a GPU. To skip it, place the fine-tuned weights in a `saved_ner_model` folder next to this notebook (see the README for how to get them), skip Section 8 and run the cell below instead.

In [15]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline
import os

# Folder containing the fine-tuned model and tokenizer
load_path = "./saved_ner_model"

try:
    print(f"Attempting to load model from: {load_path}")
    model = AutoModelForTokenClassification.from_pretrained(load_path, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(load_path, local_files_only=True)

    # Re-initialize the pipeline
    ner_pipeline = pipeline(
        "ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple"
    )
    print("Model and Tokenizer loaded successfully!")

except Exception as e:
    print(f" Load failed: {e}")
    print("\nTroubleshooting: Ensure 'saved_ner_model' folder is in the same directory as this notebook.")

Attempting to load model from: /content/drive/MyDrive/Colab Notebooks/NLP/saved_ner_model


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

Model and Tokenizer loaded successfully!


### 9. Final Evaluation on the Test Set
While the model evaluated itself on the validation set during training, standard machine learning practice requires us to report our final performance on a completely unseen `test` set to ensure the model generalizes well to new data.

In [16]:
# Evaluate the fine-tuned model on the unseen test set
from transformers import Trainer, TrainingArguments

eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./bert-ner-eval", per_device_eval_batch_size=32, report_to="none"),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Running final evaluation on the test set...")
test_results = eval_trainer.evaluate(formatted_dataset["test"])
for name in ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy"]:
    print(f"{name.replace('eval_', ''):>10}: {test_results[name]:.4f}")


Running final evaluation on the test set...


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.110953,0,0.904720,0.919618,0.912108,0.982879


 precision: 0.9047
    recall: 0.9196
        f1: 0.9121
  accuracy: 0.9829


### 10. Real-World Inference Test
To visually verify that our fine-tuned model understands context, we will pass a custom sentence through a Hugging Face `pipeline`. We will test it with a sentence containing ambiguous entities to ensure it correctly maps them based on their surrounding context.

In [17]:
# A tricky sentence with ambiguous entities (e.g., Apple the company vs. apples the fruit)
custom_sentence = "Tim Cook flew to London to announce that Apple will buy 10,000 apples from a farm in Kent."

# Run the sentence through the pipeline established in the previous section
predictions = ner_pipeline(custom_sentence)

print(f"Sentence: {custom_sentence}\n")
print("--- Model Predictions ---")
for pred in predictions:
    print(f"Entity: {pred['word']:<12} | Label: {pred['entity_group']:<5} | Confidence: {pred['score']:.4f}")

Sentence: Tim Cook flew to London to announce that Apple will buy 10,000 apples from a farm in Kent.

--- Model Predictions ---
Entity: Tim Cook     | Label: PER   | Confidence: 0.9993
Entity: London       | Label: LOC   | Confidence: 0.9992
Entity: Apple        | Label: ORG   | Confidence: 0.9836
Entity: Kent         | Label: LOC   | Confidence: 0.9990


### 11. Extended Paragraph Inference
To ensure the model is robust and can handle multi-sentence context, we will pass a full paragraph through the pipeline. This tests the model's ability to accurately tag various overlapping entities (Persons, Organizations, Locations, and Miscellaneous) within a denser linguistic structure.

In [18]:
import textwrap

# A real-world news paragraph to test extended context
news_paragraph = (
    "Andrew Bailey, the governor of the Bank of England, said the war in the Middle East "
    "is causing inflation to rise again this year. The European Central Bank also left "
    "interest rates unchanged as expected, but discussed a rate hike at length. Oil prices "
    "retreated after hitting four-year highs, and Whitbread plans to cut 3,800 jobs."
)

# Run the paragraph through the pipeline
paragraph_predictions = ner_pipeline(news_paragraph)

print("--- News Article Inference Test ---\n")

# Formatting the output for readability
wrapped_text = textwrap.fill(news_paragraph, width=80)
print(f"Text:\n{wrapped_text}\n")

print(f"{'Entity':<20} | {'Label':<10} | {'Confidence':<10}")
print("-" * 45)

for pred in paragraph_predictions:
    print(f"{pred['word']:<20} | {pred['entity_group']:<10} | {pred['score']:.4f}")

--- News Article Inference Test ---

Text:
Andrew Bailey, the governor of the Bank of England, said the war in the Middle
East is causing inflation to rise again this year. The European Central Bank
also left interest rates unchanged as expected, but discussed a rate hike at
length. Oil prices retreated after hitting four-year highs, and Whitbread plans
to cut 3,800 jobs.

Entity               | Label      | Confidence
---------------------------------------------
Andrew Bailey        | PER        | 0.9994
Bank of England      | ORG        | 0.9992
Middle East          | LOC        | 0.9984
European Central Bank | ORG        | 0.9994
Whitbread            | ORG        | 0.9656


### 12. Model Diagnostic: Manual Error Analysis Data
This section extracts specific samples from the test set where the model's predictions may deviate from the gold-standard labels. They highlight how the model handles sub-word tokenization and domain-specific shorthand (e.g., sports scorecards).

In [19]:
### 12. Model Diagnostic: Manual Error Analysis Data
# This section extracts specific samples from the test set where the model's
# predictions deviate from the gold-standard labels due to sub-word tokenization
# or domain-specific shorthand. These directly support Section 7.3 of the report.

print("--- Manual Error Analysis Diagnostics ---")
print("Extracting example test sentences...\n")

# The specific test set indices we analyzed for the report
diagnostic_indices = [21, 101]

for idx in diagnostic_indices:
    tokens = dataset["test"][idx]["tokens"]
    # Get the true string labels, ignoring the padding index if present
    true_tags = [label_list[t] for t in dataset["test"][idx]["ner_tags"]]

    sentence = " ".join(tokens)
    preds = ner_pipeline(sentence)

    print(f"Sample #{idx}")
    print(f"Original Text: {sentence}\n")

    actual_entities = [(token, tag) for token, tag in zip(tokens, true_tags) if tag != 'O']
    print(f"Gold Standard (True): {actual_entities}")

    # Show what our pipeline predicted to highlight the boundary/sub-word issues
    predicted_entities = [(p['word'], p['entity_group']) for p in preds]
    print(f"Model Predicted:      {predicted_entities}")
    print("-" * 75)

--- Manual Error Analysis Diagnostics ---
Extracting example test sentences...

Sample #21
Original Text: All four teams are level with one point each from one game .

Gold Standard (True): []
Model Predicted:      []
---------------------------------------------------------------------------
Sample #101
Original Text: Inzamamul Haq st Germon b Astle 2

Gold Standard (True): [('Inzamamul', 'B-PER'), ('Haq', 'I-PER'), ('Germon', 'B-PER'), ('Astle', 'B-PER')]
Model Predicted:      [('In', 'PER'), ('##zamamul Haq', 'PER'), ('Germon', 'PER'), ('Astle', 'PER')]
---------------------------------------------------------------------------
